# Silver Layer — Joint Angle Feature Engineering

Reads bronze (raw per-frame landmarks) and produces smoothed, phase-normalized
joint angle time series per rep.

**Schema out:** `rep_id, phase_pct, angle_name, angle_value_deg, smoothed`

Pipeline:
1. **Setup** — imports, Spark session
2. **Read Bronze** — load landmark rows for a rep
3. **Visibility filtering** — drop low-confidence frames
4. **Angle computation** — knee chamber, hip rotation, pivot foot, torso lean
5. **Savitzky-Golay smoothing** — smooth each angle series
6. **Phase normalization** — map frame timeline to 0–100 %
7. **Write Silver** — append to Delta table at `silver/joint_angles/`


## Setup

In [1]:
import os
from pathlib import Path

import numpy as np
from scipy.signal import savgol_filter
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    DoubleType,
    StringType,
    StructField,
    StructType,
)
from delta.pip_utils import configure_spark_with_delta_pip

os.environ.setdefault("SPARK_LOCAL_HOSTNAME", "localhost")
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

project_root = Path.cwd()
BRONZE_LOCATION = str(project_root / "bronze" / "pose_landmarks")
SILVER_LOCATION = str(project_root / "silver" / "joint_angles")

# Visibility threshold — frames where any required landmark is below this are dropped
VISIBILITY_THRESHOLD = 0.5

try:
    spark.stop()
except Exception:
    pass

builder = (
    SparkSession.builder
    .appName("SilverAngles")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", 8)
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("✅ SparkSession ready —", spark.version)


:: loading settings :: url = jar:file:/Users/gdoan/code/ai-micro-learn/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/gdoan/.ivy2.5.2/cache
The jars for the packages stored in: /Users/gdoan/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-61c2da94-4e02-44ab-a55c-f607df908a9d;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.delta#delta-storage;4.0.0 in central
	found org.antlr#antlr4-runtime;4.13.1 in central
:: resolution report :: resolve 144ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.13;4.0.0 from central in [default]
	io.delta#delta-storage;4.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.13.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   a

✅ SparkSession ready — 4.0.0


## Silver Schema

In [2]:
SILVER_SCHEMA = StructType([
    StructField("rep_id",          StringType(),  nullable=False),
    StructField("phase_pct",       DoubleType(),  nullable=False),  # 0.0 – 100.0
    StructField("angle_name",      StringType(),  nullable=False),
    StructField("angle_value_deg", DoubleType(),  nullable=True),
    StructField("smoothed",        BooleanType(), nullable=False),
])

# Joint angles we compute and their required landmark names.
# Each angle is the interior angle at the middle landmark (vertex).
ANGLE_DEFINITIONS: dict[str, tuple[str, str, str]] = {
    # Kicking leg
    "kick_knee_chamber":  ("right_hip",      "right_knee",  "right_ankle"),
    # Standing/pivot leg
    "pivot_foot_angle":   ("right_knee",     "right_ankle", "right_foot_index"),
    # Torso lean from vertical — approximated as shoulder-midpoint → hip-midpoint angle
    "torso_lean":         ("left_shoulder",  "right_shoulder", "right_hip"),
    # Hip rotation — angle of the hip line relative to shoulder line (horizontal plane proxy)
    "hip_rotation":       ("left_hip",       "right_hip",   "right_shoulder"),
}

print("Silver schema defined.")
print("Angles:", list(ANGLE_DEFINITIONS.keys()))


Silver schema defined.
Angles: ['kick_knee_chamber', 'pivot_foot_angle', 'torso_lean', 'hip_rotation']


## Read Bronze & Pivot to Wide Format

In [3]:
bronze_df = spark.read.format("delta").load(BRONZE_LOCATION)

# Collect all rep_ids so the caller can choose one (or loop over all).
rep_ids = [r.rep_id for r in bronze_df.select("rep_id").distinct().collect()]
print(f"Reps in bronze: {len(rep_ids)}")
for r in rep_ids:
    print(" •", r)


26/06/17 21:18:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Reps in bronze: 3
 • 01506b6f-38b5-4b64-91e0-a3abffe752a4
 • 371e49be-ce59-4b68-a0a2-5466ca1ec69a
 • b6ad799f-e338-4d8a-b2cb-ceec0dfcf747


In [4]:
# Process all reps, or set to a specific rep_id string to process just one.
TARGET_REP_ID: str | None = None  # None → process all reps

reps_to_process = rep_ids if TARGET_REP_ID is None else [TARGET_REP_ID]

def pivot_rep_to_wide(rep_id: str):
    """
    For a single rep, return a dict of:
      frame_idx -> { landmark_name: {"x": float, "y": float, "z": float, "visibility": float} }
    Only includes frames where all landmarks are present.
    Drops frames where any required landmark is below VISIBILITY_THRESHOLD.
    """
    required_landmarks = {lm for triple in ANGLE_DEFINITIONS.values() for lm in triple}

    rows = (
        bronze_df
        .filter(F.col("rep_id") == rep_id)
        .filter(F.col("landmark_name").isin(list(required_landmarks)))
        .orderBy("frame_idx", "landmark_id")
        .collect()
    )

    # Group into {frame_idx: {landmark_name: {...}}}
    frames: dict[int, dict] = {}
    for row in rows:
        frames.setdefault(row.frame_idx, {})[row.landmark_name] = {
            "x": row.x, "y": row.y, "z": row.z, "visibility": row.visibility,
        }

    # Drop frames missing any required landmark or with low visibility
    clean_frames = {
        idx: lms for idx, lms in frames.items()
        if all(
            lm in lms and (lms[lm]["visibility"] or 0.0) >= VISIBILITY_THRESHOLD
            for lm in required_landmarks
        )
    }

    dropped = len(frames) - len(clean_frames)
    print(f"  rep {rep_id[:8]}… — {len(frames)} frames, {dropped} dropped (low visibility), {len(clean_frames)} clean")
    return clean_frames


## Angle Computation, Smoothing & Phase Normalization

In [5]:
def angle_between(a: dict, b: dict, c: dict) -> float:
    """
    Interior angle at point B formed by vectors B→A and B→C.
    Uses x, y, z coordinates. Returns degrees.
    """
    ba = np.array([a["x"] - b["x"], a["y"] - b["y"], a["z"] - b["z"]])
    bc = np.array([c["x"] - b["x"], c["y"] - b["y"], c["z"] - b["z"]])
    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-9)
    return float(np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0))))


def compute_silver_rows(rep_id: str, clean_frames: dict) -> list[dict]:
    """
    For each clean frame, compute all joint angles, smooth them,
    and assign phase_pct (0–100). Returns rows matching SILVER_SCHEMA.
    """
    if not clean_frames:
        return []

    sorted_idxs = sorted(clean_frames.keys())
    n = len(sorted_idxs)

    # --- Raw angles: {angle_name: [value_per_frame]} ---
    raw: dict[str, list[float]] = {name: [] for name in ANGLE_DEFINITIONS}
    for idx in sorted_idxs:
        lms = clean_frames[idx]
        for angle_name, (lm_a, lm_b, lm_c) in ANGLE_DEFINITIONS.items():
            raw[angle_name].append(angle_between(lms[lm_a], lms[lm_b], lms[lm_c]))

    # --- Savitzky-Golay smoothing ---
    # window must be odd and <= n; fall back to raw if too few frames.
    sg_window = min(11, n if n % 2 == 1 else n - 1)
    sg_poly   = min(3, sg_window - 1)
    can_smooth = n >= sg_window

    smoothed: dict[str, list[float]] = {}
    for angle_name, values in raw.items():
        if can_smooth:
            smoothed[angle_name] = savgol_filter(values, sg_window, sg_poly).tolist()
        else:
            smoothed[angle_name] = values[:]

    # --- Phase normalization: map frame position to 0–100 % ---
    phase_pcts = [i / (n - 1) * 100.0 if n > 1 else 0.0 for i in range(n)]

    # --- Build output rows (raw + smoothed) ---
    rows: list[dict] = []
    for i, phase_pct in enumerate(phase_pcts):
        for angle_name in ANGLE_DEFINITIONS:
            # Raw row
            rows.append({
                "rep_id":          rep_id,
                "phase_pct":       round(phase_pct, 4),
                "angle_name":      angle_name,
                "angle_value_deg": raw[angle_name][i],
                "smoothed":        False,
            })
            # Smoothed row
            rows.append({
                "rep_id":          rep_id,
                "phase_pct":       round(phase_pct, 4),
                "angle_name":      angle_name,
                "angle_value_deg": smoothed[angle_name][i],
                "smoothed":        True,
            })

    return rows


print("Angle functions defined.")


Angle functions defined.


## Write Silver Delta Table

In [6]:
total_rows_written = 0

for rep_id in reps_to_process:
    print(f"\nProcessing rep: {rep_id}")
    clean_frames = pivot_rep_to_wide(rep_id)
    silver_rows  = compute_silver_rows(rep_id, clean_frames)

    if not silver_rows:
        print("  ⚠️  No clean frames — skipping.")
        continue

    df = spark.createDataFrame(silver_rows, schema=SILVER_SCHEMA)
    (
        df.write
        .format("delta")
        .mode("append")
        .save(SILVER_LOCATION)
    )
    total_rows_written += len(silver_rows)
    print(f"  ✅ Wrote {len(silver_rows)} rows ({len(silver_rows) // (2 * len(ANGLE_DEFINITIONS))} frames × {len(ANGLE_DEFINITIONS)} angles × 2 (raw+smoothed))")

print(f"\nDone. Total rows written: {total_rows_written}")
print(f"Location: {SILVER_LOCATION}")



Processing rep: 01506b6f-38b5-4b64-91e0-a3abffe752a4


  rep 01506b6f… — 61 frames, 0 dropped (low visibility), 61 clean


  ✅ Wrote 488 rows (61 frames × 4 angles × 2 (raw+smoothed))

Processing rep: 371e49be-ce59-4b68-a0a2-5466ca1ec69a
  rep 371e49be… — 21 frames, 0 dropped (low visibility), 21 clean
  ✅ Wrote 168 rows (21 frames × 4 angles × 2 (raw+smoothed))

Processing rep: b6ad799f-e338-4d8a-b2cb-ceec0dfcf747
  rep b6ad799f… — 10 frames, 0 dropped (low visibility), 10 clean
  ✅ Wrote 80 rows (10 frames × 4 angles × 2 (raw+smoothed))

Done. Total rows written: 736
Location: /Users/gdoan/code/ai-micro-learn/silver/joint_angles


## Sanity Check

In [7]:
silver_df = spark.read.format("delta").load(SILVER_LOCATION)

print(f"Total rows in silver: {silver_df.count()}")
silver_df.groupBy("rep_id", "angle_name", "smoothed").count().orderBy("rep_id", "angle_name", "smoothed").show(40, truncate=False)


Total rows in silver: 736
+------------------------------------+-----------------+--------+-----+
|rep_id                              |angle_name       |smoothed|count|
+------------------------------------+-----------------+--------+-----+
|01506b6f-38b5-4b64-91e0-a3abffe752a4|hip_rotation     |false   |61   |
|01506b6f-38b5-4b64-91e0-a3abffe752a4|hip_rotation     |true    |61   |
|01506b6f-38b5-4b64-91e0-a3abffe752a4|kick_knee_chamber|false   |61   |
|01506b6f-38b5-4b64-91e0-a3abffe752a4|kick_knee_chamber|true    |61   |
|01506b6f-38b5-4b64-91e0-a3abffe752a4|pivot_foot_angle |false   |61   |
|01506b6f-38b5-4b64-91e0-a3abffe752a4|pivot_foot_angle |true    |61   |
|01506b6f-38b5-4b64-91e0-a3abffe752a4|torso_lean       |false   |61   |
|01506b6f-38b5-4b64-91e0-a3abffe752a4|torso_lean       |true    |61   |
|371e49be-ce59-4b68-a0a2-5466ca1ec69a|hip_rotation     |false   |21   |
|371e49be-ce59-4b68-a0a2-5466ca1ec69a|hip_rotation     |true    |21   |
|371e49be-ce59-4b68-a0a2-5466ca1ec69a|